# Phase 01 — Scraping Berita 4 Bank (BCA, BRI, BNI, Mandiri) dari CNBC Indonesia

Notebook **satu fase utuh & mandiri** (self-contained) untuk mengumpulkan berita ekonomi 4 bank besar dari CNBC Indonesia.

### Metode
Mengikuti pendekatan tutorial scraping CNBC: memakai **halaman pencarian** (`/search`) dengan filter rentang tanggal (`fromdate`/`todate`), lalu untuk tiap artikel pada hasil pencarian dibuka halamannya dan diambil isinya dari `div.detail-text`. Stack: `requests` + `BeautifulSoup` + `csv` (sama seperti tutorial).

```
https://www.cnbcindonesia.com/search?query=<KEYWORD>&page=<N>&fromdate=YYYY/MM/DD&todate=YYYY/MM/DD
```

### Scope penelitian
- **Keyword:** 4 bank — tiap bank dicari dengan **nama** (BCA) **dan** kode **saham** (BBCA), lalu digabung & dedupe.
- **Periode:** 20 Oktober 2024 – 20 Oktober 2026 (dua tahun; dipakai sebagai batas waktu untuk membatasi volume data — tanpa klaim kausalitas apa pun).
- Aspek (fase berikutnya) dirujuk ke taksonomi **FiQA** (kategori Corporate & Stock).

### Prinsip
- **Isi disimpan mentah** — cleaning & pemecahan kalimat di Phase 02.
- **Checkpoint & resume** — hasil ditulis *append* ke CSV; artikel yang sudah ada dilewati saat dijalankan ulang.
- **Sopan ke server** — jeda antar permintaan + retry.
- **Pilot dulu** — `PILOT=True` menguji 1 keyword rentang pendek sebelum full-run.

> Catatan etika: data untuk riset akademik non-komersial; patuhi Ketentuan Penggunaan CNBC Indonesia.

## 1. Konfigurasi

In [ ]:
from pathlib import Path

# --- Mode ---
# PILOT=True  -> hanya 1 keyword + rentang tanggal pendek, untuk uji alur.
# PILOT=False -> semua keyword 4 bank, periode penuh.
PILOT = True

# --- Periode (format CNBC: YYYY/MM/DD) ---
FROMDATE = "2024/10/20"
TODATE   = "2026/10/20"

# Untuk PILOT: rentang pendek + 1 keyword saja
PILOT_FROMDATE = "2026/10/01"
PILOT_TODATE   = "2026/10/20"
PILOT_KEYWORD  = "BBCA"
PILOT_MAX_HAL  = 2   # batasi jumlah halaman pencarian saat pilot

# --- Keyword 4 bank: gabungan nama + kode saham ---
# Tiap entri dicari terpisah; hasil digabung & dedupe berdasarkan id artikel.
KEYWORDS = ["BCA", "BBCA", "BRI", "BBRI", "BNI", "BBNI", "Mandiri", "BMRI"]

# --- Jaringan ---
BASE = "https://www.cnbcindonesia.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}
REQUEST_DELAY = 1.0   # jeda detik antar permintaan (sopan + hindari blokir)
TIMEOUT = 30
MAX_RETRY = 3
MAX_HAL_PENGAMAN = 500  # batas atas jumlah halaman per keyword (pengaman)

# --- Penyimpanan ---
RAW_DIR = Path("data") / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
suffix = "pilot" if PILOT else "full"
OUTPUT_CSV = RAW_DIR / f"cnbc_4bank_{suffix}.csv"

# Pilih nilai efektif sesuai mode
AKT_FROM = PILOT_FROMDATE if PILOT else FROMDATE
AKT_TO   = PILOT_TODATE   if PILOT else TODATE
AKT_KEYWORDS = [PILOT_KEYWORD] if PILOT else KEYWORDS

print(f"Mode      : {'PILOT' if PILOT else 'FULL'}")
print(f"Periode   : {AKT_FROM} s/d {AKT_TO}")
print(f"Keyword   : {AKT_KEYWORDS}")
print(f"Output    : {OUTPUT_CSV}")

## 2. Utilitas jaringan & ekstraksi isi artikel

`fetch()` = unduh dengan retry. `ambil_isi_artikel()` = sama seperti tutorial: buka link artikel, ambil `div.detail-text`, gabung paragraf `<p>`.

In [ ]:
import time
import re
import requests
from bs4 import BeautifulSoup

session = requests.Session()
session.headers.update(HEADERS)

def fetch(url):
    """Unduh URL dengan retry + backoff. Return response(200) atau None."""
    for percobaan in range(1, MAX_RETRY + 1):
        try:
            r = session.get(url, timeout=TIMEOUT)
            if r.status_code == 200:
                return r
            if r.status_code in (404, 410):
                return None
        except requests.RequestException as e:
            print(f"    [ERR] {percobaan}/{MAX_RETRY}: {type(e).__name__}")
        time.sleep(2 ** percobaan)
    return None

def ambil_isi_artikel(link):
    """Buka halaman artikel -> ambil isi dari div.detail-text (gabung <p>).
    Pembersihan ringan: buang penanda embed [Gambas:...] dan kode penulis (xxx/xxx)."""
    r = fetch(link)
    if r is None:
        return ""
    soup = BeautifulSoup(r.text, "lxml")
    div = soup.find("div", class_="detail-text")
    if div is None:
        return ""
    # buang elemen non-teks + blok video sisipan yang mengotori isi
    for buang in div.find_all(["script", "style", "iframe"]):
        buang.decompose()
    for v in div.find_all("a", class_=re.compile(r"autovideocnbc")):
        v.decompose()
    paragraf = [p.get_text(" ", strip=True) for p in div.find_all("p")]
    teks = " ".join(p for p in paragraf if p)
    teks = re.sub(r"\[Gambas:[^\]]*\]", " ", teks)
    teks = re.sub(r"\(\s*[a-z]{2,4}\s*/\s*[a-z]{2,4}\s*\)\s*$", "", teks).strip()
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

## 3. Crawling halaman pencarian per keyword

Alur (mengikuti tutorial):
1. Ambil `page=1` untuk keyword tsb → cari angka halaman terbesar (`last_page`) dari link paginasi.
2. Loop `page=1..last_page` → tiap halaman `find_all('article')` → per artikel ambil **judul (`h2`), link (`a[href]`), waktu (`span` terakhir)**.
3. Buka link → ambil isi (`div.detail-text`).

Pola URL artikel dipakai untuk mengambil **id_artikel** (dedupe) dan membuang **video** (kode tipe 18/19).

In [ ]:
# Pola URL artikel CNBC: /{kanal}/{YYYYMMDDhhmmss}-{kode_tipe}-{id}/{slug}
URL_PATTERN = re.compile(r"/(\w+)/(\d{14})-(\d+)-(\d+)/")

def search_url(keyword, page, fromdate, todate):
    kw = keyword.replace(" ", "+")
    return f"{BASE}/search?query={kw}&page={page}&fromdate={fromdate}&todate={todate}"

def cari_last_page(keyword, fromdate, todate):
    """Ambil halaman 1 -> angka halaman terbesar dari link paginasi (seperti tutorial)."""
    r = fetch(search_url(keyword, 1, fromdate, todate))
    if r is None:
        return 1
    soup = BeautifulSoup(r.text, "lxml")
    angka = []
    for a in soup.find_all("a"):
        try:
            angka.append(int(a.text.strip()))
        except ValueError:
            continue
    last = max(angka) if angka else 1
    return min(last, MAX_HAL_PENGAMAN)

def parse_satu_halaman(html):
    """Dari satu halaman hasil pencarian -> list artikel teks {id, url, judul, waktu}."""
    soup = BeautifulSoup(html, "lxml")
    hasil = []
    for art in soup.find_all("article"):
        a = art.find("a", href=True)
        if not a:
            continue
        link = a["href"].split("#")[0]
        m = URL_PATTERN.search(link)
        if not m:
            continue
        kanal, ts, kode_tipe, id_artikel = m.groups()
        if kode_tipe in ("18", "19"):  # video -> buang
            continue
        # judul: h2 bila ada, fallback ke atribut alt gambar / teks anchor
        h2 = art.find("h2")
        if h2 and h2.get_text(strip=True):
            judul = h2.get_text(strip=True)
        else:
            img = a.find("img")
            judul = (img.get("alt").strip() if img and img.get("alt") else a.get_text(" ", strip=True))
        # waktu: span terakhir di dalam anchor (seperti tutorial)
        spans = a.find_all("span")
        waktu = spans[-1].get_text(strip=True) if spans else ""
        hasil.append({
            "id_artikel": id_artikel, "kanal": kanal, "url": link,
            "timestamp": ts, "judul": judul, "waktu": waktu, "kode_tipe": kode_tipe,
        })
    return hasil

## 4. Jalankan scraping (dengan checkpoint & resume)

Tiap artikel ditulis *append* ke CSV begitu selesai. `id_artikel` yang sudah ada dilewati, jadi aman dijalankan ulang bila terputus. Keyword 4 bank otomatis digabung & didedupe lewat `id_artikel`.

In [ ]:
import csv
from datetime import datetime

CSV_HEADER = ["id_artikel", "keyword", "kanal", "url", "timestamp",
              "judul", "waktu_listing", "isi_mentah", "isi_panjang", "tanggal_scrape"]

def id_sudah_ada(path):
    """Baca id_artikel yang sudah tersimpan di CSV -> set (untuk resume & dedupe)."""
    ada = set()
    if path.exists():
        with open(path, "r", encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                ada.add(str(row.get("id_artikel", "")))
    return ada

def jalankan():
    sudah = id_sudah_ada(OUTPUT_CSV)
    print(f"[RESUME] {len(sudah)} artikel sudah ada, akan dilewati.\n")

    file_baru = not OUTPUT_CSV.exists()
    f = open(OUTPUT_CSV, "a", encoding="utf-8", newline="", buffering=1)  # line-buffered
    writer = csv.DictWriter(f, fieldnames=CSV_HEADER)
    if file_baru:
        writer.writeheader()

    n_baru = 0
    try:
        for keyword in AKT_KEYWORDS:
            last_page = cari_last_page(keyword, AKT_FROM, AKT_TO)
            if PILOT:
                last_page = min(last_page, PILOT_MAX_HAL)
            print(f"=== keyword '{keyword}': {last_page} halaman ===")
            time.sleep(REQUEST_DELAY)

            for page in range(1, last_page + 1):
                r = fetch(search_url(keyword, page, AKT_FROM, AKT_TO))
                time.sleep(REQUEST_DELAY)
                if r is None:
                    print(f"  [hal {page}] gagal, lanjut")
                    continue
                artikel = parse_satu_halaman(r.text)
                for a in artikel:
                    if a["id_artikel"] in sudah:
                        continue  # sudah pernah (keyword lain / run sebelumnya)
                    isi = ambil_isi_artikel(a["url"])
                    time.sleep(REQUEST_DELAY)
                    writer.writerow({
                        "id_artikel": a["id_artikel"], "keyword": keyword, "kanal": a["kanal"],
                        "url": a["url"], "timestamp": a["timestamp"], "judul": a["judul"],
                        "waktu_listing": a["waktu"], "isi_mentah": isi, "isi_panjang": len(isi),
                        "tanggal_scrape": datetime.now().isoformat(timespec="seconds"),
                    })
                    sudah.add(a["id_artikel"])
                    n_baru += 1
                    print(f"  [{keyword} hal {page}] +{a['judul'][:60]}")
    except KeyboardInterrupt:
        print("\n[DIHENTIKAN] progres tersimpan. Jalankan ulang sel ini untuk melanjutkan.")
    finally:
        f.close()
    print(f"\n[SELESAI] {n_baru} artikel baru ditambahkan sesi ini.")

jalankan()

## 5. Verifikasi hasil

In [ ]:
import pandas as pd

if OUTPUT_CSV.exists():
    df = pd.read_csv(OUTPUT_CSV, dtype={"id_artikel": str})
    print(f"Total artikel : {len(df)}")
    print(f"id unik       : {df['id_artikel'].nunique()} (duplikat: {len(df)-df['id_artikel'].nunique()})")
    print(f"isi < 150 char: {(df['isi_panjang'] < 150).sum()}")
    print(f"rata panjang  : {df['isi_panjang'].mean():.0f} char")
    print("\nJumlah per keyword:")
    print(df["keyword"].value_counts())
    display(df[["id_artikel", "keyword", "timestamp", "isi_panjang", "judul"]].head(10))
else:
    print("Belum ada output. Jalankan sel scraping dulu.")

In [ ]:
# Lihat satu contoh isi artikel (cek kualitas ekstraksi)
if OUTPUT_CSV.exists() and len(df):
    c = df[df["isi_panjang"] >= 150].iloc[0]
    print("JUDUL :", c["judul"])
    print("URL   :", c["url"])
    print("-" * 70)
    print(c["isi_mentah"][:1200], "...")

## 6. Menjalankan full-run (4 bank, 20 Okt 2024 – 20 Okt 2026)

1. Di sel **Konfigurasi**, ubah `PILOT = False`.
2. Restart & Run All.
3. Semua keyword 4 bank (`BCA, BBCA, BRI, BBRI, BNI, BBNI, Mandiri, BMRI`) di-crawl; hasil otomatis **digabung & dedupe** berdasarkan `id_artikel`.
4. **Aman ditinggal**: bila koneksi putus / notebook ditutup, jalankan ulang sel scraping — artikel yang sudah ada di `data/raw/cnbc_4bank_full.csv` dilewati.

> Catatan: jika suatu saat halaman `/search` berhenti mengembalikan `<article>` (situs pindah ke rendering JavaScript), gunakan endpoint JSON `GET {BASE}/api/v2/search?query=<kw>&fromdate=..&todate=..&start=0&limit=100` yang isinya setara (judul, tanggal, isi) — struktur response-nya berisi field `data[].strjudul`, `data[].strisi`, `data[].dtnewsdate`, `data[].articleUrl`.